# AlexNet — From-Scratch Implementation and Analysis
### Companion notebook to the *AlexNet* lecture notes

**Graduate Deep Learning Course**

This notebook accompanies `alexnet_lecture.pdf`. It implements AlexNet's architecture
in PyTorch layer-by-layer (matching the layer table in Section 3 of the notes), verifies
the output-shape and parameter-count formulas derived in Section 4, trains a scaled-down
version on CIFAR-10 (32x32 images are upsampled to fit the receptive-field structure so
the exercise is tractable on a laptop/GPU-less machine), and visualizes learned Conv1
filters — the same qualitative check Krizhevsky et al. used in the original paper (their
Figure 3) to sanity-check that the network learns oriented edge/color-blob detectors.

**Contents**
1. Imports and reproducibility
2. Architecture implementation (`AlexNet` class)
3. Verifying output shapes and parameter counts against the lecture notes table
4. Forward pass demo on random input
5. Training loop on CIFAR-10 (small-scale, CPU-friendly settings)
6. Visualizing learned Conv1 filters
7. Exercises (code stubs matching the lecture notes exercises)


In [ ]:
# 1. Imports and reproducibility
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


## 2. Architecture Implementation

We implement AlexNet exactly as specified in Section 3 of the lecture notes:

- 5 convolutional layers (with ReLU, and LRN after Conv1/Conv2, and MaxPool after Conv1/Conv2/Conv5)
- 3 fully connected layers (with ReLU + Dropout on FC6/FC7)
- A final 1000-way softmax head (configurable `num_classes` for other datasets)

We use the **227x227** input convention (see the worked example in Section 4 of the
notes) so that the Conv1 output is exactly 55x55, matching the original paper's stated
feature-map sizes. LRN parameters (`k=2, n=5, alpha=1e-4, beta=0.75`) match Section 5.2.


In [ ]:
class AlexNet(nn.Module):
    '''AlexNet (Krizhevsky, Sutskever & Hinton, 2012).

    Matches the layer table in Section 3 of the lecture notes. Input is expected
    to be 227x227x3 (see Section 4, "Conv1 output size" worked example).
    '''

    def __init__(self, num_classes: int = 1000, dropout: float = 0.5):
        super().__init__()

        self.features = nn.Sequential(
            # Conv1: 96 x 11x11, stride 4  -> 55x55x96
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),                     # -> 27x27x96

            # Conv2: 256 x 5x5, pad 2      -> 27x27x256
            nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),                     # -> 13x13x256

            # Conv3: 384 x 3x3, pad 1      -> 13x13x384
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            # Conv4: 384 x 3x3, pad 1      -> 13x13x384
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            # Conv5: 256 x 3x3, pad 1      -> 13x13x256
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),                     # -> 6x6x256
        )

        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))  # robustness for non-227 inputs

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

        self._initialize_weights()

    def _initialize_weights(self):
        # Section 6 of the lecture notes: N(0, 0.01^2) weights;
        # bias = 1 for Conv2/4/5 and FC layers, bias = 0 elsewhere.
        bias_one_layers = set()
        conv_idx = 0
        for m in self.features:
            if isinstance(m, nn.Conv2d):
                conv_idx += 1
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
                if conv_idx in (2, 4, 5):
                    nn.init.constant_(m.bias, 1.0)
                else:
                    nn.init.constant_(m.bias, 0.0)
        for m in self.classifier:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.01)
                nn.init.constant_(m.bias, 1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


model = AlexNet(num_classes=1000).to(device)
print(model)


## 3. Verifying Shapes and Parameter Counts

We register forward hooks to print the output shape after every layer, then compare
against the layer table in Section 3 of the notes. We also recompute the total
parameter count and confirm the ~62M figure (and the ~95% fully-connected share
mentioned in the notes).


In [ ]:
def print_layer_shapes(model, input_size=(1, 3, 227, 227)):
    x = torch.randn(*input_size).to(device)
    print(f"Input:  {tuple(x.shape)}")
    for i, layer in enumerate(model.features):
        x = layer(x)
        print(f"features[{i}] {layer.__class__.__name__:<20s} -> {tuple(x.shape)}")
    x = model.avgpool(x)
    print(f"avgpool                          -> {tuple(x.shape)}")
    x = torch.flatten(x, 1)
    print(f"flatten                          -> {tuple(x.shape)}")
    for i, layer in enumerate(model.classifier):
        x = layer(x)
        print(f"classifier[{i}] {layer.__class__.__name__:<20s} -> {tuple(x.shape)}")
    return x

with torch.no_grad():
    out = print_layer_shapes(model)
print(f"\\nFinal output shape: {tuple(out.shape)}  (expected: (1, 1000))")


In [ ]:
def count_parameters(model):
    conv_params = sum(p.numel() for name, p in model.named_parameters() if "features" in name)
    fc_params = sum(p.numel() for name, p in model.named_parameters() if "classifier" in name)
    total = conv_params + fc_params
    return conv_params, fc_params, total

conv_params, fc_params, total_params = count_parameters(model)
print(f"Convolutional params: {conv_params:>12,}")
print(f"Fully connected params: {fc_params:>10,}")
print(f"Total params:          {total_params:>12,}")
print(f"FC share of total:      {100 * fc_params / total_params:>10.1f}%")
print("\\n(Compare to Section 3 of the lecture notes: ~62M total, ~95% in FC layers.)")


In [ ]:
# Sanity check against the closed-form conv parameter formula from Section 4:
#   Params = C_out * (C_in * F^2 + 1)
def conv_param_formula(c_in, c_out, f):
    return c_out * (c_in * f * f + 1)

# Conv2: C_in=96, C_out=256, F=5  -> should equal 614,656 (worked example in the notes)
formula_result = conv_param_formula(96, 256, 5)
actual_result = model.features[4].weight.numel() + model.features[4].bias.numel()
print(f"Conv2 params (formula): {formula_result:,}")
print(f"Conv2 params (actual):  {actual_result:,}")
assert formula_result == actual_result, "Mismatch between formula and actual layer!"
print("Match confirmed.")


## 4. Forward Pass Demo

A quick end-to-end check that a batch of random "images" flows through the network
and produces class probabilities.


In [ ]:
model.eval()
dummy_batch = torch.randn(4, 3, 227, 227).to(device)
with torch.no_grad():
    logits = model(dummy_batch)
    probs = F.softmax(logits, dim=1)

print(f"Logits shape: {tuple(logits.shape)}")
print(f"Probabilities sum to 1 per row: {torch.allclose(probs.sum(dim=1), torch.ones(4).to(device), atol=1e-5)}")
print(f"Top-1 predicted class per sample: {probs.argmax(dim=1).tolist()}")


## 5. Training on CIFAR-10 (Small-Scale, CPU-Friendly)

Training the full ImageNet-scale AlexNet (Section 6 of the notes: ~90 epochs, 2x GTX
580, 5-6 days) is not practical in a teaching notebook. Instead we:

- Use CIFAR-10 (10 classes, 32x32 images) resized up to 64x64 (not the full 227x227,
  to keep this runnable on CPU in minutes rather than hours),
- Shrink the classifier width (`fc_dim`) so the model trains quickly while keeping the
  same conv/pool/LRN/dropout structure,
- Reproduce the AlexNet-style optimizer settings from Section 6: SGD, momentum 0.9,
  weight decay 5e-4, and step-based learning-rate decay.

Students should treat this as a structural / methodological exercise, not a claim
that this reproduces the paper's ImageNet numbers.


In [ ]:
class AlexNetSmall(nn.Module):
    '''A CIFAR-10-friendly AlexNet variant: same layer structure and design
    choices (ReLU, LRN, overlapping max-pool, dropout) as the full model, but
    sized for 64x64 input and 10 classes so it trains quickly for teaching.'''

    def __init__(self, num_classes: int = 10, dropout: float = 0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2),   # 64x64x64
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),                  # 31x31x64

            nn.Conv2d(64, 192, kernel_size=5, stride=1, padding=2), # 31x31x192
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),                  # 15x15x192

            nn.Conv2d(192, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),                  # 7x7x256
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(256 * 6 * 6, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(1024, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


In [ ]:
# Data pipeline: CIFAR-10, resized to 64x64, with AlexNet-style augmentation
# (random crop + horizontal flip -- see Section 5.5 of the lecture notes)
transform_train = transforms.Compose([
    transforms.Resize(72),
    transforms.RandomCrop(64),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

# Set download=True the first time you run this notebook with internet access.
train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

classes = train_set.classes
print(f"Train set size: {len(train_set)}, Test set size: {len(test_set)}, Classes: {classes}")


In [ ]:
small_model = AlexNetSmall(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
# Section 6 hyperparameters: SGD, momentum 0.9, weight decay 5e-4
optimizer = optim.SGD(small_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100.0 * correct / total


In [ ]:
NUM_EPOCHS = 3  # increase for a real training run; kept small for a fast teaching demo

history = {"train_loss": [], "test_acc": []}

for epoch in range(NUM_EPOCHS):
    small_model.train()
    running_loss = 0.0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = small_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    test_acc = evaluate(small_model, test_loader)
    history["train_loss"].append(avg_loss)
    history["test_acc"].append(test_acc)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train loss: {avg_loss:.4f} | test acc: {test_acc:.2f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], marker="o", color="#0B1F3A")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")

axes[1].plot(history["test_acc"], marker="o", color="#B8862B")
axes[1].set_title("Test Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
plt.tight_layout()
plt.show()


## 6. Visualizing Learned Conv1 Filters

The original paper's Figure 3 shows the 96 learned Conv1 kernels split across the two
GPUs, with one GPU specializing in color-agnostic edge/frequency detectors and the
other in color-blob detectors. We reproduce the same qualitative visualization here
for our (much smaller) Conv1 layer.


In [ ]:
def visualize_conv1_filters(model, layer_index=0, max_filters=64):
    conv1 = model.features[layer_index]
    weights = conv1.weight.data.cpu().clone()

    # Normalize each filter to [0, 1] for display
    w_min, w_max = weights.min(), weights.max()
    weights = (weights - w_min) / (w_max - w_min + 1e-8)

    n_filters = min(max_filters, weights.shape[0])
    grid_size = int(np.ceil(np.sqrt(n_filters)))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        if i < n_filters:
            filt = weights[i].permute(1, 2, 0).numpy()
            ax.imshow(filt)
        ax.axis("off")
    plt.suptitle("Learned Conv1 Filters (early training)", fontsize=12)
    plt.tight_layout()
    plt.show()

visualize_conv1_filters(small_model, layer_index=0)
print("Note: after only a few epochs these filters are still close to their random")
print("initialization. Train for more epochs to see oriented edge/color-blob structure emerge.")


## 7. Exercises (Code Stubs)

These mirror the "Exercises" section of the lecture notes. Fill in each function.


In [ ]:
def exercise_1_pool5_output_size():
    '''Exercise 1: output size of Pool5 given Conv5 output 13x13x256,
    kernel 3x3, stride 2, no padding. Use the formula from Section 4 of the notes:
        W_out = floor((W - F + 2P) / S) + 1
    Return the integer spatial size (should match the 6x6 used in AlexNet.forward).
    '''
    W, F, S, P = 13, 3, 2, 0
    # TODO: implement using the formula above
    raise NotImplementedError


def exercise_2_fc6_params():
    '''Exercise 2: number of parameters in FC6, given Pool5 output flattened
    to 9216 (=6*6*256) and FC6 has 4096 units. Include the bias term.
    '''
    # TODO: implement using Params = C_out * (C_in + 1) for a dense layer
    raise NotImplementedError


def exercise_3_augmentation_multiplier():
    '''Exercise 3: effective dataset size multiplier from random 224x224 crops
    of a 256x256 image, plus horizontal flipping. Return the integer multiplier.
    '''
    # TODO: count distinct crop offsets in each spatial dimension, square it,
    # then account for the horizontal flip.
    raise NotImplementedError


# Quick self-check once implemented:
# assert exercise_1_pool5_output_size() == 6
# assert exercise_2_fc6_params() == 37_752_832
# assert exercise_3_augmentation_multiplier() == 2048


---
### References
1. A. Krizhevsky, I. Sutskever, and G. E. Hinton, *ImageNet Classification with Deep Convolutional Neural Networks*, NeurIPS 2012.
2. See `alexnet_lecture.pdf` for the full derivations referenced throughout this notebook.
